# **Previsão de Churn (Abandono) em Telecom (MLP - MLPClassifier (sklearn))**

# 1. Importando bibliotecas

In [ ]:
import sys

def in_colab():
    return 'google.colab' in sys.modules

In [ ]:
if in_colab():
    !pip install optuna
    !pip install tensorflow
    !pip install matplotlib_venn

In [ ]:
# Sistema e utilitários
import random
from copy import deepcopy
import warnings

# Bibliotecas principais
import pandas as pd
import numpy as np
import optuna

# Matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Circle, Rectangle
from matplotlib_venn import venn2

# Scikit-learn
from sklearn.metrics import accuracy_score, log_loss, classification_report, mean_squared_error
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve
from sklearn.exceptions import ConvergenceWarning
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.utils import shuffle

In [ ]:
# Remove warnings:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# 2. Importando banco de dados


In [ ]:
# Importar banco de dados
if in_colab():
    data = pd.read_csv("customer_churn_telecom_services.csv")
else:
    data = pd.read_csv("../customer_churn_telecom_services.csv")
data.head()

# 3. Dividindo dados

In [ ]:
def separa_classe(df_class):
    df_class = shuffle(df_class, random_state=42)

    n = len(df_class)
    train = df_class.iloc[:int(0.5*n)]
    val   = df_class.iloc[int(0.5*n):int(0.75*n)]
    test  = df_class.iloc[int(0.75*n):]

    return train, val, test


def balanceamento(df_maior, df_menor):
    df_menor_bal = df_menor.sample(n=len(df_maior), replace=True, random_state=42)
    return df_menor_bal


def junta_e_mistura(df_x, df_y):
  return shuffle(pd.concat([df_x, df_y]), random_state=42)

In [ ]:
# Separar classes
# Classe 0 = Churn no
# Classe 1 = Churn yes
data_cl0 = data[data["Churn"] == "No"]
data_cl1 = data[data["Churn"] == "Yes"]

# Para cada classe, separar treino(50%), validação(25%) e teste(25%) de cada classe
data_cl0_train, data_cl0_val, data_cl0_test = separa_classe(data_cl0)
data_cl1_train, data_cl1_val, data_cl1_test = separa_classe(data_cl1)

# Balancear os dados de treino e validação da classe 2 até ter mesma quantidade que os da classe 1
data_cl1_train = balanceamento(data_cl0_train, data_cl1_train)
data_cl1_val = balanceamento(data_cl0_val, data_cl1_val)


# Embaralhar dados de treino, validação e teste
data_train = junta_e_mistura(data_cl0_train, data_cl1_train)
data_val = junta_e_mistura(data_cl0_val, data_cl1_val)
data_test = junta_e_mistura(data_cl0_test, data_cl1_test)

In [ ]:
print("Tamanhos finais:")
print("Treino:", data_train.shape)
print("Validação:", data_val.shape)
print("Teste:", data_test.shape)

# 4. Tratamento de dados

In [ ]:
# ===============================
# Funções de pré-processamento
# ===============================
def treat_missing(df):
    df = df.copy()

    # Strings vazias ou só espaços → NaN
    df = df.replace(r"^\s*$", np.nan, regex=True)

    print("Linhas removidas:", df.isna().any(axis=1).sum())

    # Remove linhas com qualquer NaN
    df = df.dropna().reset_index(drop=True)

    return df


def fix_inconsistencies(df):
    df = df.copy()

    # 1. TotalCharges (string -> numeric)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    return df


def treating_data(df):
    df = df.copy()

    # ===============================
    # 1. Binárias Yes / No (TODAS)
    # ===============================
    yes_no_cols = [
        "Partner", "Dependents", "PhoneService",
        "PaperlessBilling", "MultipleLines",
        "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport",
        "StreamingTV", "StreamingMovies"
    ]
    for col in yes_no_cols:
        if col in df.columns:
            df[col] = (df[col] == "Yes").astype(int)

    # ===============================
    # 2. Gender
    # ===============================
    df["gender"] = (df["gender"] == "Male").astype(int)

    # ===============================
    # 3. One-Hot Encoding
    # ===============================
    cat_cols = [
        "InternetService",
        "Contract",
        "PaymentMethod"
    ]
    df = pd.get_dummies(
        df,
        columns=cat_cols,
        drop_first=False,
        dtype=int
    )

    # ===============================
    # 4. Agregação de serviços (remove ruído)
    # ===============================
    service_cols = [
        "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport",
        "StreamingTV", "StreamingMovies"
    ]

    df["num_services"] = df[service_cols].sum(axis=1)
    df.drop(columns=service_cols, inplace=True)

    return df


def new_features(df):
    df["AvgMonthlyCharge"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["is_new_customer"] = (df["tenure"] <= 6).astype(int)
    return df


def is_binary(col):
    return set(col.dropna().unique()) <= {0, 1}

In [ ]:
# 0. Copia banco de dados
tratando_train = data_train.copy()
tratando_val = data_val.copy()
tratando_test = data_test.copy()

In [ ]:
#1. Trata dados faltando:
tratando_train = treat_missing(tratando_train)
tratando_val = treat_missing(tratando_val)
tratando_test = treat_missing(tratando_test)

In [ ]:
# 2. Muda target de "Yes"/"No" para 1/0:
tratando_train["Churn"] = (tratando_train["Churn"] == "Yes").astype(int)
tratando_val["Churn"] = (tratando_val["Churn"] == "Yes").astype(int)
tratando_test["Churn"] = (tratando_test["Churn"] == "Yes").astype(int)

In [ ]:
# 3. Fix Inconsistências
tratando_train = fix_inconsistencies(tratando_train)
tratando_val = fix_inconsistencies(tratando_val)
tratando_test = fix_inconsistencies(tratando_test)

In [ ]:
# 4. Tratando dados:
tratando_train = treating_data(tratando_train)
tratando_val = treating_data(tratando_val)
tratando_test = treating_data(tratando_test)

In [ ]:
# 5. Add novas features:
tratando_train = new_features(tratando_train)
tratando_val = new_features(tratando_val)
tratando_test = new_features(tratando_test)

In [ ]:
# 6. Separa valores target do resto do datasets:
target = "Churn"

# - Separa valores target de treino, validação e teste:
y_train = tratando_train[target]
y_val   = tratando_val[target]
y_test  = tratando_test[target]

# - Remover os valores target dos datasets para passar ao preprocessor:
tratando_train = tratando_train.drop(columns=[target])
tratando_val = tratando_val.drop(columns=[target])
tratando_test = tratando_test.drop(columns=[target])

In [ ]:
# 7. Normalização
# Separa colunas entree categoricas e numericas:
binary_cols = [
    c for c in tratando_train.columns
    if c != target and is_binary(tratando_train[c])
]

numeric_cols = [
    c for c in tratando_train.columns
    if c != target and c not in binary_cols
]

# Cria preprocessor:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("bin", "passthrough", binary_cols)
    ]
)

# Fit no treino:
preprocessor.fit(tratando_train)

# Transform no resto:
x_train = preprocessor.transform(tratando_train)
x_val   = preprocessor.transform(tratando_val)
x_test  = preprocessor.transform(tratando_test)

In [ ]:
print("Formato dos dados:")
print("Dados completos:", x_train)
print("Shape:", x_train.shape)
print("Uma linha:", x_train[0])
print("Tipo:", type(x_train))

# 5. Hiperparametros

In [ ]:
hidden_layers_limits = (2, 3)
layer_sizes_limits = (32, 256)
learning_rate_limits = (1e-6, 1e-3)
activation_functions = ["relu", "tanh", "logistic"]
optimizators = ["lbfgs", "sgd", "adam"]
L2_regularization_limits = (1e-6, 1e-3)
batch_size_limits = (30, 300)
epochs = 10000
patience = 20
random_state = random.randint(0, 99999)
threshold = 0.5
trials_optuna = 40

# 6. Treinamento

In [ ]:
def fix_random_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)

fix_random_seed(random_state)
print("random state:", random_state)

In [ ]:
# Calcula KS:
def calcula_ks(y_true, y_proba, plot=False):
    # y_proba deve ser prob da classe positiva (1)
    probs = y_proba
    labels = y_true

    # Dividir probabilidades por classe
    probs_0 = np.sort(probs[labels == 0])
    probs_1 = np.sort(probs[labels == 1])

    # CDFs (acumuladas)
    cdf_0 = np.arange(1, len(probs_0) + 1) / len(probs_0)
    cdf_1 = np.arange(1, len(probs_1) + 1) / len(probs_1)

    # Emparelhamento por grade comum de thresholds
    all_probs = np.sort(np.unique(probs))
    cdf0_interp = np.interp(all_probs, probs_0, cdf_0)
    cdf1_interp = np.interp(all_probs, probs_1, cdf_1)

    # KS por diferença máxima
    ks_values = np.abs(cdf1_interp - cdf0_interp)
    ks_stat = np.max(ks_values)
    best_threshold = all_probs[np.argmax(ks_values)]

    if plot:
        # Resultados:
        print("KS Statistic:", ks_stat)
        print("Best threshold:", best_threshold)
        # Grafico:
        plt.figure(figsize=(7, 5))
        plt.plot(all_probs, cdf0_interp, label="Class 0", linewidth=2)
        plt.plot(all_probs, cdf1_interp, label="Class 1", linewidth=2)

        # Linha pontilhada do KS
        plt.plot([best_threshold, best_threshold],
                [cdf0_interp[np.argmax(ks_values)], cdf1_interp[np.argmax(ks_values)]],
                'k--', linewidth=2, label=f'KS = {ks_stat:.3f} em {best_threshold:.3f}')

        plt.title("KS Statistic Plot")
        plt.xlabel("Threshold")
        plt.ylabel("Percentage below threshold")

        plt.legend()
        plt.grid(True)
        plt.show()
    return ks_stat, best_threshold

In [ ]:
# Processo de treino do optuna
def objective(trial):
  # Hiperparâmetros escolhidos automaticamente:

  hidden_layers_number = trial.suggest_int("hidden_layers", hidden_layers_limits[0], hidden_layers_limits[-1])
  layers = []
  for i in range(hidden_layers_number):
    units_in_layer =  trial.suggest_int(f"units_in_layer_{i}", layer_sizes_limits[0], layer_sizes_limits[-1])
    layers.append(units_in_layer)

  params = {
      "hidden_layer_sizes": layers,
      "learning_rate_init": trial.suggest_float("learning_rate_init", learning_rate_limits[0], learning_rate_limits[-1], log=True),
      "activation": trial.suggest_categorical("activation", activation_functions),
      "solver": trial.suggest_categorical("solver", optimizators),
      "alpha": trial.suggest_float("alpha", L2_regularization_limits[0], L2_regularization_limits[-1], log=True),
      "batch_size": trial.suggest_int("batch_size", batch_size_limits[0], batch_size_limits[-1]),
      "max_iter": 1,
      "warm_start": True,
      "random_state": random_state
      }

  # Modelo:
  model = MLPClassifier(**params)

  # Treino:
  wait = 0
  best_val_loss = np.inf

  for epoch in range(epochs):
    # Treinando
    model.fit(x_train, y_train)

    # Validando
    pred_proba = model.predict_proba(x_val)[:, 1]
    current_val_loss = log_loss(y_val, pred_proba)

    # Verifica se houve ou não uma melhora na loss
    if current_val_loss < best_val_loss:
      best_val_loss = current_val_loss
      wait = 0
    else:
      wait += 1

    # Early stop
    if wait >= patience:
      break
    
  # Calcula ks:
  pred_prob = model.predict(x_val).ravel()
  best_ks, _ = calcula_ks(y_val, pred_prob)

  return best_ks

In [ ]:
# Optuna executando busca por melhores hiperparametros
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=trials_optuna)

In [ ]:
# Resultado do treino com optuna
print("Melhores parâmetros:", study.best_params)
print("Melhor loss:", study.best_value)

In [ ]:
# Treinamento do Modelo final usando os melhores hiperparametros

# Melhores hiperparametros:
best_params = study.best_params
#best_params = {'hidden_layers': 3, 'units_in_layer_0': 143, 'units_in_layer_1': 16, 'units_in_layer_2': 84, 'learning_rate_init': 0.006917822802406749, 'activation': 'tanh', 'solver': 'lbfgs', 'alpha': 0.00010191560773786016, 'batch_size': 68}


# Reconstruir arquitetura:
n_layers = best_params["hidden_layers"]
layer_sizes = []
for i in range(n_layers):
  layer_sizes.append(best_params[f"units_in_layer_{i}"])

params = {
    "hidden_layer_sizes": layer_sizes,
    "learning_rate_init": best_params["learning_rate_init"],
    "activation": best_params["activation"],
    "solver": best_params["solver"],
    "alpha": best_params["alpha"],
    "batch_size": best_params["batch_size"],
    "max_iter": 1,
    "warm_start": True,
    "random_state": random_state
    }

# Cria modelo final:
final_model = MLPClassifier(**params)

In [ ]:
#Treinamento do modelo final
wait = 0
best_val_loss = np.inf
best_model = None

for epoch in range(epochs):
  # Treinando
  final_model.fit(x_train, y_train)

  # Validando
  pred_proba = final_model.predict_proba(x_val)[:, 1]
  current_val_loss = log_loss(y_val, pred_proba)

  # Verifica se houve ou não uma melhora na loss
  if current_val_loss < best_val_loss:
    best_val_loss = current_val_loss
    best_model = deepcopy(final_model)
    wait = 0
  else:
    wait += 1

  print(f"Epoch: {epoch}/{epochs} --- Current validation loss: {current_val_loss} --- Best validation loss: {best_val_loss} --- Wait for new loss: {patience - wait}")

  # Early stop
  if wait >= patience:
    print("Early stop ativado")
    break

# Modelo com melhor desempenho durante o treino
final_model = best_model

# 7. Analize de desempenho

In [ ]:
def predict_com_threshold(model, data, threshold=0.5):
    proba = model.predict_proba(data)[:, 1]
    pred = (proba >= threshold).astype(int)
    return pred

In [ ]:
# === 1. Obter previsões (treino, validação e teste) ===
y_train_pred = predict_com_threshold(final_model, x_train, threshold)
y_val_pred = predict_com_threshold(final_model, x_val, threshold)
y_test_pred = predict_com_threshold(final_model, x_test, threshold)


y_train_proba = final_model.predict_proba(x_train)[:, 1]
y_val_proba = final_model.predict_proba(x_val)[:, 1]
y_test_proba = final_model.predict_proba(x_test)[:, 1]

In [ ]:
# === 2. Relatórios de classificação (treino, validação, teste) ===
print("==== CLASSIFICATION REPORT - TREINO ====")
print(classification_report(y_train, y_train_pred))
print(f"Accuracy no treino: {accuracy_score(y_train, y_train_pred):.4f}")
print()

print("==== CLASSIFICATION REPORT - VALIDAÇÃO ====")
print(classification_report(y_val, y_val_pred))
print(f"Accuracy na validação: {accuracy_score(y_val, y_val_pred):.4f}")
print()

print("==== CLASSIFICATION REPORT - TESTE ====")
print(classification_report(y_test, y_test_pred))
print(f"Accuracy no teste: {accuracy_score(y_test, y_test_pred):.4f}")

In [ ]:
# === 3. Precision, Recall e F1 ===
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("F1-Score:", f1_score(y_test, y_test_pred))

In [ ]:
# === 4. MSE (Erro Médio Quadrado) ===
print("MSE Treino:", mean_squared_error(y_train, y_train_pred))
print("MSE Validação:", mean_squared_error(y_val, y_val_pred))
print("MSE Teste:", mean_squared_error(y_test, y_test_pred))

In [ ]:
# === 5. Entropia Cruzada (Log Loss) ===
print("Log Loss Treino:", log_loss(y_train, y_train_proba))
print("Log Loss Validação:", log_loss(y_val, y_val_proba))
print("Log Loss Teste:", log_loss(y_test, y_test_proba))

In [ ]:
# === 6. Matriz de Confusão ===
fig, ax = plt.subplots(figsize=(6,6))
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, ax=ax)
plt.title("Matriz de Confusão - Teste")
plt.show()

In [ ]:
# === 7. Diagrama TN, FP, FN, TP ===
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()

fig, ax = plt.subplots(figsize=(7,7))
ax.set_aspect('equal')
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.axis('off')

# ==========================================
# 1) FUNDOS (fora do círculo)
# ==========================================

# Esquerda (False Negatives)
bg_left  = Rectangle((-1.1, -1.1), 1.1, 2.2, facecolor='#e7f3e8', edgecolor='none', zorder=0)

# Direita (True Negatives)
bg_right = Rectangle((0.0, -1.1), 1.1, 2.2, facecolor='#efeded', edgecolor='none', zorder=0)

ax.add_patch(bg_left)
ax.add_patch(bg_right)

# ==========================================
# 2) METADE INTERNA DO CÍRCULO (TP e FP)
# ==========================================

radius = 0.75
center = (0, 0)

# TP = metade esquerda interna
tp_wedge = Wedge(center, radius, 90, 270, facecolor='#cff7cc', edgecolor='none', zorder=1)
# FP = metade direita interna
fp_wedge = Wedge(center, radius, 270, 90, facecolor='#ffd7d9', edgecolor='none', zorder=1)

ax.add_patch(tp_wedge)
ax.add_patch(fp_wedge)

# Contorno do círculo
ax.add_patch(Circle(center, radius, facecolor='none', edgecolor='black', lw=2.2, zorder=3))

# ==========================================
# 3) TEXTOS
# ==========================================

# ---- Regiões externas ----
# False Negatives (topo esquerdo)
ax.text(-0.55, 0.95, "False Negatives", fontsize=16, ha='center')
ax.text(-0.55, 0.80, f"FN = {fn}", fontsize=14, ha='center')

# True Negatives (topo direito)
ax.text(0.55, 0.95, "True Negatives", fontsize=16, ha='center')
ax.text(0.55, 0.80, f"TN = {tn}", fontsize=14, ha='center')

# ---- Regiões internas ----
ax.text(-0.35, 0.05, "True Positives", fontsize=16, ha='center')
ax.text(-0.35, -0.18, f"TP = {tp}", fontsize=14, ha='center')

ax.text( 0.35, 0.05, "False Positives", fontsize=16, ha='center')
ax.text( 0.35, -0.18, f"FP = {fp}", fontsize=14, ha='center')

# Texto inferior
ax.text(0, -1.0, "Retrieved elements", fontsize=14, ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# === 8. Teste Kolmogorov-Smirnov (KS) ===
calcula_ks(y_test, y_test_proba, True)

In [ ]:
# === 9. AUROC ===
auc = roc_auc_score(y_test, y_test_proba)
print("AUC (Área sob a ROC):", auc)

In [ ]:
# === 9.5. Curva ROC ===
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr)
plt.title("Curva ROC")
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.show()

In [ ]:
# === 10. Curva Precision-Recall ===
prec, rec, thr = precision_recall_curve(y_test, y_test_proba)
plt.figure(figsize=(8,6))
plt.plot(rec, prec)
plt.title("Curva Precision-Recall")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

In [ ]:
# === 11. Distribuição das probabilidades ===
plt.figure(figsize=(8,6))
plt.hist(y_test_proba[y_test==0], bins=30, alpha=0.6, label='Classe 0')
plt.hist(y_test_proba[y_test==1], bins=30, alpha=0.6, label='Classe 1')
plt.title("Distribuição das Probabilidades - Teste")
plt.xlabel("Probabilidade prevista")
plt.ylabel("Frequência")
plt.legend()
plt.show()